In [ ]:
#Environment Initialization & Library Setup
#Purpose: Initialize the core processing environments and ensure all vector, raster, and machine learning packages are loaded cleanly.

# Package Ingestion Matrix
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, LineString, Polygon
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

# Verify deep learning availability
try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except ImportError:
    YOLO_AVAILABLE = False
    print("⚠️ Warning: ultralytics package not found. YOLOv8 steps will run in simulation mode.")

# Configure runtime workspace folders
DIRS = ["../data/raw", "../data/processed", "../data/vectors", "../models", "../runs"]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

print("✅ Environment initialized. Storage directories verified.")

✅ Environment initialized. Storage directories verified.


In [20]:
# Step 1 - Spatial & Raster Preprocessing (Member A)
#Member A Vector Processing Engine
def execute_vector_preprocessing(input_shapefile, output_vector_path, buffer_meters=60):
    print("🔄 [Member A] Slicing country-wide waterways layer...")

# 1. Establish absolute bounding box for Kasarani District, Nairobi
    kasarani_bbox = box(36.80, -1.32, 36.95, -1.20)

    # If a real shapefile isn't found locally, generate a mock dataset to keep the pipeline functional
    if not os.path.exists(input_shapefile):
        print("⚠️ Real shapefile not found. Generating production-grade mock river tracking vector...")
        mock_line = LineString([(36.8219, -1.2921), (36.8350, -1.2850), (36.8500, -1.2900)])
        raw_gdf = gpd.GeoDataFrame(geometry=[mock_line], crs="EPSG:4326")
        raw_gdf['fclass'] = 'river'
    else:
        # Load only the features intersecting our target area to protect system RAM
        raw_gdf = gpd.read_file(input_shapefile, bbox=kasarani_bbox)


 # 2. Filter out non-river features
    if 'fclass' in raw_gdf.columns:
        filtered_rivers = raw_gdf[raw_gdf['fclass'].isin(['river', 'stream'])].copy()
    else:
        filtered_rivers = raw_gdf.copy()